# Self Query Retriever — LLM 기반 메타데이터 필터링

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **SelfQueryRetriever**가 자연어 쿼리를 검색어와 메타데이터 필터로 자동 분리하는 원리 이해하기
- `AttributeInfo`로 메타데이터 스키마를 정의하는 방법 익히기
- 날짜, 카테고리, 키워드 등 다양한 필터 조건으로 검색하기

## 사전 지식

- 메타데이터(Metadata): 문서에 첨부된 구조화된 속성값 (year, category 등)
- VectorStoreRetriever 기본 사용법

---

## 핵심 아이디어

### 일반 검색과의 차이

| 방식 | 처리 과정 |
|------|----------|
| **일반 검색** | "2020년 이후 딥러닝 논문" → 전체 텍스트로 벡터 검색 |
| **Self Query** | "2020년 이후 딥러닝 논문" → 검색어: "딥러닝 논문" + 필터: `year >= 2020` |

LLM이 자연어를 분석해 검색어와 메타데이터 필터를 자동으로 분리합니다.

> **주의**: SelfQueryRetriever는 Chroma, Pinecone 등 일부 벡터스토어만 공식 지원해요. FAISS는 지원 범위가 제한적입니다.

> **실무 팁**: `verbose=True`로 설정하면 LLM이 생성한 필터 조건을 콘솔에서 확인할 수 있어서 디버깅에 유용해요.

In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
# ---------------------------------------------------
# 메타데이터가 포함된 샘플 문서를 Chroma에 저장
# ---------------------------------------------------

# ============================================================
# TODO: category, year, topic 메타데이터를 가진 5개 문서를 Chroma에 저장하세요
# 힌트: Document(page_content=..., metadata={"category": ..., "year": ..., "topic": ...})
# 예상 결과: 문서 5개 저장 완료 메시지 출력
# ============================================================

from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.schema import AttributeInfo
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

# 메타데이터가 있는 샘플 문서

docs = [
    Document(
        page_content="Scikit-learn은 2007년에 시작된 Python 기계 학습 라이브러리입니다.",
        metadata={"category": "라이브러리", "year": 2007, "topic": "기계학습"}
    ),
    Document(
        page_content="Transformer는 2017년 발표된 혁신적인 딥러닝 아키텍처입니다.",
        metadata={"category": "모델", "year": 2017, "topic": "딥러닝"}
    ),
    Document(
        page_content="Word2Vec은 2013년 Google에서 개발한 단어 임베딩 기법입니다.",
        metadata={"category": "기법", "year": 2013, "topic": "NLP"}
    ),
    Document(
        page_content="GPT-3는 2020년에 출시된 대규모 언어 모델입니다.",
        metadata={"category": "모델", "year": 2020, "topic": "언어모델"}
    ),
    Document(
        page_content="HuggingFace는 2016년 설립된 NLP 플랫폼입니다.",
        metadata={"category": "플랫폼", "year": 2016, "topic": "NLP"}
    ),
]

# Vectorstore 생성 (SelfQuery에서 공식 지원하는 Chroma 사용)
# Chroma: SelfQueryRetriever가 메타데이터 필터를 직접 전달할 수 있는 벡터스토어
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(docs, embeddings)

# Vectorstore 생성 (SelfQuery에서 공식 지원하는 Chroma 사용)
# Chroma: SelfQueryRetriever가 메타데이터 필터를 직접 전달할 수 있는 벡터스토어


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [3]:
# ---------------------------------------------------
# AttributeInfo와 SelfQueryRetriever 생성
# ---------------------------------------------------

# ============================================================
# TODO: AttributeInfo로 category, year, topic 필드를 정의하고 SelfQueryRetriever를 생성하세요
# 힌트: AttributeInfo(name, description, type) — type은 "string" 또는 "integer"
# 예상 결과: SelfQueryRetriever 생성 완료 메시지 출력
# ============================================================

# 메타데이터 필드 정의
# AttributeInfo: LLM에게 메타데이터 필드의 의미와 타입을 알려주는 스키마
from tabnanny import verbose


metadata_field_info = [
    AttributeInfo(
        name="category",
        description="문서의 카테고리(라이브러리, 모델, 플랫폼)",
        type="string"
    ),
    AttributeInfo(
        name="year",
        description="문서가 만들어지거나 발표된 연도",
        type="integer"
    ),
    AttributeInfo(
        name="topic",
        description="주제 (기계학습, 딥러닝, NLP, 언어모델)",
        type="string"
    ),
]

# 문서 컨텐츠 설명
documents_content_description = "AI 기술, 라이브러리, 모델에 대한 설명"

# LLM
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

# SelfQueryRetriever 생성
# verbose=True: 생성된 구조화 쿼리를 콘솔에서 확인 가능 (디버깅에 유용)
self_query_retriever = SelfQueryRetriever.from_llm(
    llm,
    vectorstore,
    document_contents=documents_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True
)


In [ ]:
# 테스트 쿼리

query = "2015년 이후 발표된 모델에 대해 알려줘."
docs = self_query_retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\n[문서 {i}]")
    print(f"내용: {doc.page_content}")
    print(f"메타데이터: {doc.metadata}")

structured_query = self_query_retriever.query_constructor.invoke(query)
print(f' ==> [Line 11]: \033[38;2;21;204;62m[structured_query]\033[0m({type(structured_query).__name__}) = \033[38;2;54;68;10m{structured_query}\033[0m')



[문서 1]
내용: HuggingFace는 2016년 설립된 NLP 플랫폼입니다.
메타데이터: {'category': '플랫폼', 'topic': 'NLP', 'year': 2016}

[문서 2]
내용: Transformer는 2017년 발표된 혁신적인 딥러닝 아키텍처입니다.
메타데이터: {'category': '모델', 'topic': '딥러닝', 'year': 2017}

[문서 3]
내용: GPT-3는 2020년에 출시된 대규모 언어 모델입니다.
메타데이터: {'category': '모델', 'topic': '언어모델', 'year': 2020}
 ==> [Line 11]: [structured_query](StructuredQuery) = query=' ' filter=Comparison(comparator=<Comparator.GT: 'gt'>, attribute='year', value=2015) limit=None


---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 자연어 → LLM이 의미 검색 쿼리 + 메타데이터 필터 자동 분리 |
| 필수 설정 | `AttributeInfo` 스키마 정의 (필드명, 타입, 설명) |
| 지원 VectorStore | Chroma, Pinecone, Weaviate 등 (FAISS 미지원) |
| 디버깅 방법 | `verbose=True` 설정 시 생성된 구조화 쿼리 확인 가능 |
| LLM 의존성 | 자연어 파싱에 LLM 호출 필요 — 응답 품질이 LLM 성능에 달림 |

### AttributeInfo 타입 지원 범위

| 데이터 타입 | 예시 | 필터 연산자 |
|-------------|------|-------------|
| `string` | 장르, 언어 | `eq`, `ne`, `in` |
| `integer` | 연도, 페이지 수 | `eq`, `gt`, `lt`, `gte`, `lte` |
| `float` | 평점, 가격 | `eq`, `gt`, `lt`, `gte`, `lte` |
| `list[string]` | 태그, 저자 목록 | `contain` |

### 다음 노트북 예고

**09-TimeWeightedVectorStoreRetriever** — 벡터 유사도와 최신성(시간 가중치)을 결합해, 오래된 문서보다 최신 문서를 우선 반환하는 검색기를 배워요.